In [2]:
!pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [3]:
%cd ../
import torch
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn as nn

from models.yolo import Model


/home/su_kark/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/su_kark/Documents/Master Research Project/quantized-yolov5


In [4]:
ckpt = torch.load('./runs/train/exp/weights/best.pt', map_location='cpu')

print(len(ckpt))
for k, v in ckpt.items():
    print(k)
    if 'scaling' in k:
        print(v)


67
model.0.conv.weight
model.0.bn.weight
model.0.bn.bias
model.0.bn.running_mean
model.0.bn.running_var
model.0.bn.num_batches_tracked
model.0.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.58594, dtype=torch.float16)
model.2.conv.weight
model.2.bn.weight
model.2.bn.bias
model.2.bn.running_mean
model.2.bn.running_var
model.2.bn.num_batches_tracked
model.2.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.58594, dtype=torch.float16)
model.4.conv.weight
model.4.bn.weight
model.4.bn.bias
model.4.bn.running_mean
model.4.bn.running_var
model.4.bn.num_batches_tracked
model.4.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.58594, dtype=torch.float16)
model.6.conv.weight
model.6.bn.weight
model.6.bn.bias
model.6.bn.running_mean
model.6.bn.running_var
model.6.bn.num_batches_tracked
model.6.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.58594, dtype=torc

/tmp/ipykernel_2532936/1623036458.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('./runs/train/exp/weights/best.pt', map_location='cpu')


In [5]:
cfg = 'models/hub/yolov1-tiny-quant.yaml'

model = Model(cfg)
model.load_state_dict(ckpt)
model.to("cpu")

/home/su_kark/.local/lib/python3.10/site-packages/torch/_tensor.py:1413: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1925.)
  return super().rename(names)
/home/su_kark/.local/lib/python3.10/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Reversing anchor order


Model(
  (model): Sequential(
    (0): QuantConv(
      (conv): QuantConv2d(
        3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (input_quant): ActQuantProxyFromInjector(
          (_zero_hw_sentinel): StatelessBuffer()
        )
        (output_quant): ActQuantProxyFromInjector(
          (_zero_hw_sentinel): StatelessBuffer()
        )
        (weight_quant): WeightQuantProxyFromInjector(
          (_zero_hw_sentinel): StatelessBuffer()
          (tensor_quant): RescalingIntQuant(
            (int_quant): IntQuant(
              (float_to_int_impl): RoundSte()
              (tensor_clamp_impl): TensorClampSte()
              (delay_wrapper): DelayWrapper(
                (delay_impl): _NoDelay()
              )
            )
            (scaling_impl): StatsFromParameterScaling(
              (parameter_list_stats): _ParameterListStats(
                (first_tracked_param): _ViewParameterWrapper(
                  (view_shape_impl): OverOutputChannelV

In [6]:
image = "/home/su_kark/Documents/Master Research Project/datasets/hpd2_416_augmented/test/images/img_0107_jpg.rf.fe620b41ce8f044692281de1021cb868.jpg"

# Load image
img = Image.open(image).convert("RGB")  # ensures 3 channels
img_array = np.array(img, dtype=np.float32)  # (H, W, C)

# Rearrange to (C, H, W)
img_array = np.transpose(img_array, (2, 0, 1))

# Add batch dimension → (B, C, H, W)
img_array = np.expand_dims(img_array, axis=0)
output_path = "input_image.npy"

# Save as .npy
np.save(output_path, img_array)

print(f"Saved image as {output_path} with shape {img_array.shape}")


Saved image as input_image.npy with shape (1, 3, 416, 416)


In [7]:
test_input_np = np.load("input_image.npy")
test_input = torch.from_numpy(test_input_np).float() / 255.0

In [8]:
print('INPUT:', test_input.shape)
print(test_input)

INPUT: torch.Size([1, 3, 416, 416])
tensor([[[[0.19608, 0.13333, 0.11373,  ..., 0.89020, 0.90980, 0.89804],
          [0.36471, 0.26667, 0.23529,  ..., 0.27843, 0.31765, 0.32549],
          [0.26275, 0.21176, 0.28627,  ..., 0.09412, 0.12941, 0.15294],
          ...,
          [0.01176, 0.01176, 0.01176,  ..., 0.18039, 0.12549, 0.07451],
          [0.01176, 0.01176, 0.01176,  ..., 0.14902, 0.09020, 0.03529],
          [0.01176, 0.01176, 0.01176,  ..., 0.12549, 0.06275, 0.00784]],

         [[0.19608, 0.13333, 0.11373,  ..., 0.89020, 0.90980, 0.89804],
          [0.36471, 0.26667, 0.23529,  ..., 0.27843, 0.31765, 0.32549],
          [0.26275, 0.21176, 0.28627,  ..., 0.09412, 0.12941, 0.15294],
          ...,
          [0.01176, 0.01176, 0.01176,  ..., 0.18039, 0.12549, 0.07451],
          [0.01176, 0.01176, 0.01176,  ..., 0.14902, 0.09020, 0.03529],
          [0.01176, 0.01176, 0.01176,  ..., 0.12549, 0.06275, 0.00784]],

         [[0.19608, 0.13333, 0.11373,  ..., 0.89020, 0.90980, 0.89

In [9]:
# QONNX VERIFICATION
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.onnx_exec import execute_onnx

In [10]:
model_file_path = "./runs/train/exp/weights/best.qonnx.onnx"
qonnx_cleanup(model_file_path, out_file=model_file_path)

In [11]:
input_dict = {"global_in": test_input_np.astype(np.float32) / 255.0}

In [12]:
model = ModelWrapper(model_file_path)
output_dict = execute_onnx(model, input_dict, return_full_exec_context=True)

In [13]:
print("OUTPUT KEYS:", output_dict.keys())

OUTPUT KEYS: dict_keys(['global_in', 'global_out', 'Quant_0_out0', 'Quant_10_out0', 'Quant_1_out0', 'Quant_11_out0', 'Quant_2_out0', 'Quant_12_out0', 'Quant_3_out0', 'Quant_13_out0', 'Quant_4_out0', 'Quant_14_out0', 'Quant_5_out0', 'Quant_15_out0', 'Quant_6_out0', 'Quant_16_out0', 'Quant_7_out0', 'Quant_17_out0', 'Quant_8_out0', 'Quant_18_out0', 'Quant_9_out0', 'Quant_0_param0', 'Quant_0_param1', 'Quant_0_param2', 'Quant_0_param3', 'Quant_10_param0', 'Quant_1_param3', 'Quant_1_param0', 'Quant_1_param1', 'Quant_2_param0', 'Quant_2_param1', 'Quant_3_param0', 'Quant_3_param1', 'Quant_4_param0', 'Quant_4_param1', 'Quant_5_param0', 'Quant_5_param1', 'Quant_6_param0', 'Quant_6_param1', 'Quant_7_param0', 'Quant_7_param1', 'Quant_8_param0', 'Quant_8_param1', 'Quant_9_param0', 'Quant_9_param1', 'Div_0_param0', 'Quant_19_param0', 'Conv_0_out0', 'BatchNormalization_0_out0', 'Relu_0_out0', 'MaxPool_0_out0', 'Conv_1_out0', 'BatchNormalization_1_out0', 'Relu_1_out0', 'MaxPool_1_out0', 'Conv_2_out0',

In [14]:
print(output_dict['global_out'].shape)

(1, 18, 13, 13)


In [15]:
class Detect(nn.Module):
    stride = [32]  # strides computed during build
    onnx_dynamic = False  # ONNX export parameter

    def __init__(self, nc=80, anchors=(), ch=(), inplace=True, in_conv=False):  # detection layer
        super().__init__()
        self.nc = nc  # number of classes
        self.no = nc + 5  # number of outputs per anchor
        self.nl = len(anchors)  # number of detection layers
        self.na = len(anchors[0]) // 2  # number of anchors
        self.grid = [torch.zeros(1)] * self.nl  # init grid
        self.anchor_grid = [torch.zeros(1)] * self.nl  # init anchor grid
        self.register_buffer('anchors', torch.tensor(anchors).float().view(self.nl, -1, 2))  # shape(nl,na,2)
        self.m = nn.ModuleList(nn.Conv2d(x, self.no * self.na, 1) for x in ch)  # output conv
        self.inplace = inplace  # use in-place ops (e.g. slice assignment)
        self.in_conv = in_conv
        self.training = False

    def forward(self, x):
        z = []  # inference output
        for i in range(self.nl):
            if self.in_conv:
                x[i] = self.m[i](x[i])  # conv
            bs, _, ny, nx = x[i].shape  # x(bs,255,20,20) to x(bs,3,20,20,85)
            x[i] = x[i].view(bs, self.na, self.no, ny, nx).permute(0, 1, 3, 4, 2).contiguous()

            if not self.training:  # inference
                if self.onnx_dynamic or self.grid[i].shape[2:4] != x[i].shape[2:4]:
                    self.grid[i], self.anchor_grid[i] = self._make_grid(nx, ny, i)

                y = x[i]
                y[..., 0:2] = (y[..., 0:2] + 0.5 + self.grid[i]) * self.stride[i]  # xy
                y[..., 2:4] = (y[..., 2:4] + 1.0) ** 2 * self.anchor_grid[i]  # wh
                y[..., 4:] = y[..., 4:] / 2 + 0.5
                z.append(y.view(bs, -1, self.no))

        return x if self.training else (torch.cat(z, 1), x)

    def _make_grid(self, nx=20, ny=20, i=0):
        d = self.anchors[i].device
        if check_version(torch.__version__, '1.10.0'):  # torch>=1.10.0 meshgrid workaround for torch>=0.7 compatibility
            yv, xv = torch.meshgrid([torch.arange(ny).to(d), torch.arange(nx).to(d)], indexing='ij')
        else:
            yv, xv = torch.meshgrid([torch.arange(ny).to(d), torch.arange(nx).to(d)])
        grid = torch.stack((xv, yv), 2).expand((1, self.na, ny, nx, 2)).float()
        anchor_grid = (self.anchors[i].clone() * self.stride[i]) \
            .view((1, self.na, 1, 1, 2)).expand((1, self.na, ny, nx, 2)).float()
        return grid, anchor_grid
   

In [16]:
names = ['person']
nc  = 1
anchors = np.array([[10,14,23,27,37,58]]) / np.array([32])
scale = np.load("./runs/train/exp/weights/best.npy")
detect_head = Detect(nc, anchors)

In [17]:
print(scale)

0.007874016


In [18]:
import cv2
from utils.utils import letterbox, non_max_suppression, scale_coords, check_version, visualize_boxes

img_org = cv2.imread(image)
img = img_org.copy()

h, w, _ = img_org.shape
img, ratio, (dw, dh) = letterbox(img, (416,416), auto=False)

img = img[:, :, ::-1]
img = np.transpose(img, (2, 0, 1))
img = np.expand_dims(img, 0) / 255.0
img = np.array(img, dtype=np.float32)
print(img.shape)

(1, 3, 416, 416)


In [ ]:
input_dict = {"global_in": img}

output_dict = execute_onnx(model, input_dict, return_full_exec_context=True)
output = output_dict['global_out']
output = scale*output
print(output.shape)
        
output = torch.from_numpy(output)

(1, 18, 13, 13)


In [27]:
pred = detect_head([output])[0]
print(pred.shape)

torch.Size([1, 507, 6])


In [28]:
# Get detected boxes_detected, labels, confidences, class-scores.
pred = non_max_suppression(pred, conf_thres=0.02, iou_thres=0.10, classes=None, max_det=1000)

In [29]:
boxes_detected, class_names_detected, probs_detected = [], [], []
# Process predictions
for i, det in enumerate(pred):  # per image
    if len(det):
        # Rescale boxes from img_size to im0 size
        det[:, :4] = scale_coords(img.shape, det[:, :4], img_org.shape).round()
        # Print results
        for c in np.unique(det[:, -1]):
            n = (det[:, -1] == c).sum()  # detections per class
            print(f"{n} {names[int(c)]}")  # add to string
        # Write results
        for *xyxy, conf, cls in reversed(det):
            c = int(cls)  # integer class
            label = f'{names[c]} {conf:.2f}'
            boxes_detected.append(xyxy)
            class_names_detected.append(names[c])
            probs_detected.append(conf)

4 person


In [30]:
# Visualize.
image_boxes = visualize_boxes(img_org, boxes_detected, class_names_detected, probs_detected, name_bgr_dict = {'person':(0,255,0)})
cv2.imwrite(output_path+f".jpg", image_boxes)

plt.imshow(cv2.cvtColor(image_boxes, cv2.COLOR_BGR2RGB))
plt.show()

{'person': (0, 255, 0)}
{'person': (0, 255, 0)}
{'person': (0, 255, 0)}
{'person': (0, 255, 0)}
